# LeetCode 1859 — Sorting the Sentence bằng Python

Link đề: https://leetcode.com/problems/sorting-the-sentence/

Độ khó: **Easy** · Chủ đề: String, Sorting

Skeleton chữ ký LeetCode:

```python
class Solution:
    def sortSentence(self, s: str) -> str:

```

**Mục tiêu học**

1. Tách câu thành từ bằng `split()`, ghép lại bằng `' '.join(...)`.
2. Dùng indexing/slicing trên chuỗi: `w[-1]` (ký tự cuối), `w[:-1]` (bỏ ký tự cuối).
3. Học 4 cách: **Bubble Sort** tự viết → `sorted(..., key=...)` → **đặt thẳng vào vị trí** (bucket) `O(n)` → dict vị trí.
4. Hiểu vì sao "đặt vào vị trí" không cần so sánh mà vẫn đúng.
5. Trace, Big-O, lỗi hay gặp và 6 bài tập về sắp xếp theo khóa.

## 1. Bài toán yêu cầu gì?

Một **câu** là các từ cách nhau **đúng một dấu cách**, không có dấu cách thừa ở đầu/cuối. Mỗi từ gồm chữ hoa/thường tiếng Anh.

Câu được **xáo trộn** bằng cách: gắn **số thứ tự (bắt đầu từ 1)** vào cuối mỗi từ, rồi đổi chỗ các từ.

Cho câu đã xáo trộn `s` (tối đa **9** từ), hãy khôi phục câu gốc.

| s | Kết quả |
|---|---|
| `"is2 sentence4 This1 a3"` | `"This is a sentence"` |
| `"Myself2 Me1 I4 and3"` | `"Me Myself and I"` |

**Constraints:** `2 <= len(s) <= 200`; chỉ chữ cái, dấu cách, chữ số 1..9; số từ từ 1 đến 9; cách nhau một dấu cách.

Vì có tối đa 9 từ, số thứ tự luôn là **một chữ số** — chính là ký tự cuối của mỗi từ.

In [1]:
examples = [("is2 sentence4 This1 a3", "This is a sentence"), ("Myself2 Me1 I4 and3", "Me Myself and I")]
for s, expected in examples:
    print(repr(s), "->", repr(expected))

'is2 sentence4 This1 a3' -> 'This is a sentence'
'Myself2 Me1 I4 and3' -> 'Me Myself and I'


## 2. Nền tảng Python

### 2.1. `split()` và `' '.join(...)`

- `s.split()` tách theo khoảng trắng, trả về **list các từ**.
- `' '.join(words)` nối các từ lại, chen **một dấu cách** giữa chúng.

In [2]:
s = "is2 sentence4 This1 a3"
words = s.split()
print(words)
print(" ".join(["This", "is", "a", "sentence"]))
print("-".join(words))

['is2', 'sentence4', 'This1', 'a3']
This is a sentence
is2-sentence4-This1-a3


### 2.2. Ký tự cuối `w[-1]` và phần còn lại `w[:-1]`

- `w[-1]`: ký tự **cuối**.
- `w[:-1]`: từ đầu tới **trước** ký tự cuối — tức bỏ chữ số.

In [3]:
w = "sentence4"
print("w[-1]  =", repr(w[-1]))
print("w[:-1] =", repr(w[:-1]))
print("int(w[-1]) =", int(w[-1]), type(int(w[-1])))

w[-1]  = '4'
w[:-1] = 'sentence'
int(w[-1]) = 4 <class 'int'>


### 2.3. Chuỗi `'4'` khác số `4`

So sánh chuỗi chữ số một ký tự `'1' < '2' < ... < '9'` vẫn đúng thứ tự, nhưng để dùng làm **chỉ số list** ta phải đổi sang `int`. Nhớ: thứ tự trong đề bắt đầu từ **1**, chỉ số list bắt đầu từ **0** ⇒ vị trí = `int(w[-1]) - 1`.

In [4]:
print("'3' < '4':", "3" < "4")
print("'10' < '9':", "10" < "9", "(so sánh chuỗi từng ký tự — may là đề chỉ có 1..9)")
for w in ["This1", "is2", "a3"]:
    print(w, "-> vị trí list =", int(w[-1]) - 1)

'3' < '4': True
'10' < '9': True (so sánh chuỗi từng ký tự — may là đề chỉ có 1..9)
This1 -> vị trí list = 0
is2 -> vị trí list = 1
a3 -> vị trí list = 2


### 2.4. `sorted(list, key=hàm)`

`sorted` trả về list mới đã sắp xếp. Tham số `key` nhận một **hàm** áp lên từng phần tử; Python sắp theo giá trị hàm trả về. `lambda w: w[-1]` nghĩa là "sắp theo ký tự cuối".

In [5]:
words = ["is2", "sentence4", "This1", "a3"]
print(sorted(words))                          # mặc định: theo bảng chữ cái (chữ hoa trước)
print(sorted(words, key=lambda w: w[-1]))     # theo ký tự cuối
print(sorted(["bb", "a", "ccc"], key=len))    # key có thể là hàm có sẵn như len

['This1', 'a3', 'is2', 'sentence4']
['This1', 'is2', 'a3', 'sentence4']
['a', 'bb', 'ccc']


Chú ý dòng đầu: `"This1"` đứng trước vì chữ hoa `'T'` có mã nhỏ hơn chữ thường — sắp xếp mặc định **không** giúp gì cho bài này.

## 3. Cách 1 — Bubble Sort tự viết (theo tag của đề)

Ý tưởng **Bubble Sort**: đi qua list, so sánh **hai phần tử kề nhau**; nếu sai thứ tự thì **đổi chỗ**. Lặp lại tới khi không còn cặp sai. Phần tử "lớn" nổi dần về cuối như bọt khí.

Đổi chỗ trong Python: `a[i], a[i+1] = a[i+1], a[i]`.

In [6]:
a = [3, 1, 2]
a[0], a[1] = a[1], a[0]
print("sau khi đổi a[0], a[1]:", a)

sau khi đổi a[0], a[1]: [1, 3, 2]


In [7]:
def sort_sentence_bubble(s: str) -> str:
    words = s.split()
    n = len(words)
    for end in range(n - 1, 0, -1):          # phần cuối dần dần đã đúng chỗ
        swapped = False
        for i in range(end):
            if int(words[i][-1]) > int(words[i + 1][-1]):
                words[i], words[i + 1] = words[i + 1], words[i]
                swapped = True
        if not swapped:                       # không đổi gì -> đã sắp xong
            break
    return " ".join(w[:-1] for w in words)

for s, expected in examples:
    print(sort_sentence_bubble(s), "| expected", expected)
    assert sort_sentence_bubble(s) == expected

This is a sentence | expected This is a sentence
Me Myself and I | expected Me Myself and I


`words[i][-1]` = ký tự cuối của từ thứ `i`. Biến `swapped` giúp dừng sớm khi list đã có thứ tự.

### 3.1. Nhận xét

Bubble Sort tốn `O(n²)` phép so sánh. Với `n <= 9` thì không sao, nhưng ta đang **dùng sắp xếp so sánh** cho dữ liệu mà ta **đã biết sẵn vị trí đích** của từng phần tử! Đó là gợi ý cho cách nhanh hơn.

## 4. Cách 2 — `sorted` với `key`

Python có sẵn thuật toán sắp xếp tốt (Timsort, `O(n log n)`). Ta chỉ cần nói "sắp theo số cuối".

In [8]:
def sort_sentence_sorted(s: str) -> str:
    words = sorted(s.split(), key=lambda w: int(w[-1]))
    return " ".join(w[:-1] for w in words)

for s, expected in examples:
    assert sort_sentence_sorted(s) == expected
print(sort_sentence_sorted("Myself2 Me1 I4 and3"))

Me Myself and I


`" ".join(w[:-1] for w in words)` dùng **generator expression**: với mỗi từ, cắt bỏ chữ số rồi đưa thẳng vào `join`.

### 4.1. Insight — mỗi từ tự mang "địa chỉ" của nó

Chữ số cuối **chính là vị trí** của từ trong câu gốc. Vì vậy không cần so sánh từ nào với từ nào: chuẩn bị `n` ô trống, rồi **đặt mỗi từ thẳng vào ô `int(w[-1]) - 1`**.

Giống phát bài kiểm tra: mỗi tờ có số báo danh, ta đặt tờ vào đúng chồng số đó — không cần so hai tờ với nhau.

In [9]:
words = "is2 sentence4 This1 a3".split()
slots = [None] * len(words)
print("ô trống:", slots)
for w in words:
    pos = int(w[-1]) - 1
    slots[pos] = w[:-1]
    print(f"đặt {w!r:<12} vào ô {pos} -> {slots}")

ô trống: [None, None, None, None]
đặt 'is2'        vào ô 1 -> [None, 'is', None, None]
đặt 'sentence4'  vào ô 3 -> [None, 'is', None, 'sentence']
đặt 'This1'      vào ô 0 -> ['This', 'is', None, 'sentence']
đặt 'a3'         vào ô 2 -> ['This', 'is', 'a', 'sentence']


`[None] * n` tạo list `n` ô, mỗi ô là `None` (giá trị "chưa có gì" trong Python).

## 5. Cách 3 — lời giải chính: đặt thẳng vào vị trí

**Đoạn 1:** `words = s.split()`.

**Đoạn 2:** `res = [""] * len(words)` — các ô trống.

**Đoạn 3:** với mỗi `w`: `res[int(w[-1]) - 1] = w[:-1]`.

**Đoạn 4:** `return " ".join(res)`.

In [10]:
def sort_sentence_place(s: str) -> str:
    words = s.split()
    res = [""] * len(words)
    for w in words:
        res[int(w[-1]) - 1] = w[:-1]
    return " ".join(res)

for s, expected in examples:
    assert sort_sentence_place(s) == expected
print(sort_sentence_place("is2 sentence4 This1 a3"))

This is a sentence


## 6. TRACE từng bước

Trace `s = "Myself2 Me1 I4 and3"`:

| bước | w | w[-1] | vị trí = int−1 | w[:-1] | res sau bước |
|---:|---|:---:|---:|---|---|
| 1 | Myself2 | 2 | 1 | Myself | `['', 'Myself', '', '']` |
| 2 | Me1 | 1 | 0 | Me | `['Me', 'Myself', '', '']` |
| 3 | I4 | 4 | 3 | I | `['Me', 'Myself', '', 'I']` |
| 4 | and3 | 3 | 2 | and | `['Me', 'Myself', 'and', 'I']` |

Kết quả: `"Me Myself and I"`.

In [11]:
def trace_sort_sentence(s: str) -> str:
    words = s.split()
    print("words =", words)
    res = [""] * len(words)
    for step, w in enumerate(words, start=1):
        digit = w[-1]
        pos = int(digit) - 1
        res[pos] = w[:-1]
        print(f"bước {step}: w={w!r:<10} w[-1]={digit!r} -> ô {pos}, đặt {w[:-1]!r:<8} res={res}")
    out = " ".join(res)
    print("kết quả:", repr(out))
    return out

assert trace_sort_sentence("Myself2 Me1 I4 and3") == "Me Myself and I"

words = ['Myself2', 'Me1', 'I4', 'and3']
bước 1: w='Myself2'  w[-1]='2' -> ô 1, đặt 'Myself' res=['', 'Myself', '', '']
bước 2: w='Me1'      w[-1]='1' -> ô 0, đặt 'Me'     res=['Me', 'Myself', '', '']
bước 3: w='I4'       w[-1]='4' -> ô 3, đặt 'I'      res=['Me', 'Myself', '', 'I']
bước 4: w='and3'     w[-1]='3' -> ô 2, đặt 'and'    res=['Me', 'Myself', 'and', 'I']
kết quả: 'Me Myself and I'


`enumerate(words, start=1)` đếm từ 1 thay vì 0 — tiện cho cột "bước".

Trace thứ hai: câu chỉ một từ, và câu 9 từ đảo ngược hoàn toàn.

In [12]:
assert trace_sort_sentence("Hello1") == "Hello"
print()
s9 = " ".join(f"w{chr(97 + i)}{i + 1}" for i in range(9)[::-1])
print("input:", s9)
assert sort_sentence_place(s9) == " ".join(f"w{chr(97 + i)}" for i in range(9))
print("output:", sort_sentence_place(s9))

words = ['Hello1']
bước 1: w='Hello1'   w[-1]='1' -> ô 0, đặt 'Hello'  res=['Hello']
kết quả: 'Hello'

input: wi9 wh8 wg7 wf6 we5 wd4 wc3 wb2 wa1
output: wa wb wc wd we wf wg wh wi


## 7. Cách 4 — dict vị trí → từ

Biến thể của cách 3: lưu vào `dict` `{vị_trí: từ}`, rồi đọc theo `1..n`. Hữu ích khi số thứ tự **không liên tục** hoặc không biết trước số từ.

In [13]:
def sort_sentence_dict(s: str) -> str:
    pos = {}
    for w in s.split():
        pos[int(w[-1])] = w[:-1]
    return " ".join(pos[k] for k in sorted(pos))

print(sort_sentence_dict("is2 sentence4 This1 a3"))
print({int(w[-1]): w[:-1] for w in "is2 sentence4 This1 a3".split()})

This is a sentence
{2: 'is', 4: 'sentence', 1: 'This', 3: 'a'}


## 8. Code nộp LeetCode

### Cách A — đặt thẳng vào vị trí (khuyên dùng)

In [14]:
class Solution:
    def sortSentence(self, s: str) -> str:
        words = s.split()
        res = [""] * len(words)
        for w in words:
            res[int(w[-1]) - 1] = w[:-1]
        return " ".join(res)

print(Solution().sortSentence("is2 sentence4 This1 a3"))

This is a sentence


### Cách B — `sorted` theo ký tự cuối (một dòng)

In [15]:
class SolutionSorted:
    def sortSentence(self, s: str) -> str:
        return " ".join(w[:-1] for w in sorted(s.split(), key=lambda w: w[-1]))

print(SolutionSorted().sortSentence("Myself2 Me1 I4 and3"))

Me Myself and I


Ở đây key là **chuỗi** `w[-1]` — vẫn đúng vì chỉ có một chữ số `'1'..'9'`, và so sánh chuỗi một ký tự trùng thứ tự số.

### Cách C — Bubble Sort

In [16]:
class SolutionBubble:
    def sortSentence(self, s: str) -> str:
        words = s.split()
        n = len(words)
        for end in range(n - 1, 0, -1):
            for i in range(end):
                if words[i][-1] > words[i + 1][-1]:
                    words[i], words[i + 1] = words[i + 1], words[i]
        return " ".join(w[:-1] for w in words)

print(SolutionBubble().sortSentence("Myself2 Me1 I4 and3"))

Me Myself and I


## 9. Vì sao đúng?

Gọi câu gốc là `o_1 o_2 ... o_n`. Khi xáo trộn, từ `o_k` trở thành `o_k + str(k)` và bị đặt ở vị trí bất kỳ.

**Cách đặt vào vị trí:** với mỗi từ đã xáo `w = o_k + str(k)`, ta có `w[-1] = str(k)` (vì `k <= 9` nên đúng một chữ số) và `w[:-1] = o_k` (vì từ gốc chỉ có chữ cái, không có chữ số). Ta đặt `o_k` vào ô `k - 1`. Mỗi `k` trong `1..n` xuất hiện **đúng một lần**, nên mọi ô được lấp đúng một lần ⇒ `res = [o_1, ..., o_n]`.

**Cách sắp xếp:** sắp theo khóa `k` cho đúng thứ tự `1, 2, ..., n` (các khóa khác nhau nên không có hòa).

**Bubble Sort:** sau vòng ngoài thứ `t`, `t` phần tử có khóa lớn nhất đã nằm đúng ở cuối (invariant của Bubble Sort); sau `n - 1` vòng, toàn bộ đã đúng.

## 10. Độ phức tạp

Gọi `n` = số từ (≤ 9), `L = len(s)` (≤ 200).

| Cách | Time | Space |
|---|---|---|
| Bubble Sort | `O(n² + L)` | `O(L)` |
| `sorted` | `O(n log n + L)` | `O(L)` |
| Đặt vào vị trí | `O(L)` | `O(L)` |
| Dict | `O(n log n + L)` (do `sorted(pos)`) | `O(L)` |

`split` và `join` đều tốn `O(L)` vì phải đọc/ghi mọi ký tự. Với `n <= 9`, mọi cách đều tức thì; điểm học là **khi đã biết vị trí đích thì không cần so sánh**.

## 11. Bộ test

Nhóm: mẫu đề; một từ; đã đúng thứ tự; đảo ngược; 9 từ; chữ hoa/thường lẫn lộn; ngẫu nhiên đối chiếu mọi cách (sinh câu gốc → xáo → khôi phục).

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<18} PASS ({actual!r})")

sol = Solution()
check("mẫu 1", sol.sortSentence("is2 sentence4 This1 a3"), "This is a sentence")
check("mẫu 2", sol.sortSentence("Myself2 Me1 I4 and3"), "Me Myself and I")
check("một từ", sol.sortSentence("Hi1"), "Hi")
check("đã đúng thứ tự", sol.sortSentence("a1 b2 c3"), "a b c")
check("đảo ngược", sol.sortSentence("c3 b2 a1"), "a b c")
check("hoa thường", sol.sortSentence("zZ2 Aa1"), "Aa zZ")

mẫu 1              PASS ('This is a sentence')
mẫu 2              PASS ('Me Myself and I')
một từ             PASS ('Hi')
đã đúng thứ tự     PASS ('a b c')
đảo ngược          PASS ('a b c')
hoa thường         PASS ('Aa zZ')


In [18]:
import random
import string
rng = random.Random(1859)

def shuffle_sentence(original_words):
    tagged = [w + str(i + 1) for i, w in enumerate(original_words)]
    rng.shuffle(tagged)
    return " ".join(tagged)

impls = [sort_sentence_bubble, sort_sentence_sorted, sort_sentence_place, sort_sentence_dict,
         Solution().sortSentence, SolutionSorted().sortSentence, SolutionBubble().sortSentence]
for _ in range(1000):
    n = rng.randint(1, 9)
    original = ["".join(rng.choice(string.ascii_letters) for _ in range(rng.randint(1, 8))) for _ in range(n)]
    s = shuffle_sentence(original)
    expected = " ".join(original)
    for f in impls:
        assert f(s) == expected, (f.__name__ if hasattr(f, "__name__") else f, s)
print("7 cài đặt khôi phục đúng 1000 câu ngẫu nhiên PASS")

7 cài đặt khôi phục đúng 1000 câu ngẫu nhiên PASS


`rng.shuffle(list)` xáo trộn list **tại chỗ**. `string.ascii_letters` = `a..zA..Z`. Test này **tự tạo dữ liệu đúng đề** rồi kiểm ta khôi phục được câu gốc — một kiểu test "khứ hồi" (round-trip).

## 12. Lỗi người mới hay gặp

### Lỗi 1: quên bỏ chữ số

In [19]:
def wrong_keep_digit(s):
    return " ".join(sorted(s.split(), key=lambda w: w[-1]))

print("sai :", repr(wrong_keep_digit("is2 This1")), "| đúng:", repr(Solution().sortSentence("is2 This1")))

sai : 'This1 is2' | đúng: 'This is'


Sửa: `w[:-1]` khi ghép.

### Lỗi 2: quên trừ 1 (thứ tự đề bắt đầu từ 1)

In [20]:
def wrong_no_minus_one(s):
    words = s.split()
    res = [""] * len(words)
    for w in words:
        res[int(w[-1])] = w[:-1]
    return " ".join(res)

try:
    wrong_no_minus_one("is2 This1")
except IndexError as e:
    print("Lỗi:", e, "-> vị trí 2 vượt list 2 ô; phải dùng int(w[-1]) - 1")

Lỗi: list assignment index out of range -> vị trí 2 vượt list 2 ô; phải dùng int(w[-1]) - 1


### Lỗi 3: sắp xếp mặc định (theo bảng chữ cái)

In [21]:
def wrong_default_sort(s):
    return " ".join(w[:-1] for w in sorted(s.split()))

print("sai :", wrong_default_sort("is2 sentence4 This1 a3"), "| đúng:", Solution().sortSentence("is2 sentence4 This1 a3"))

sai : This a is sentence | đúng: This is a sentence


Sắp xếp theo **cả từ**, không theo số. Sửa: `key=lambda w: w[-1]`.

### Lỗi 4: `split(" ")` + ghép bằng `+` để lại dấu cách thừa

In [22]:
def wrong_concat(s):
    words = sorted(s.split(" "), key=lambda w: w[-1])
    out = ""
    for w in words:
        out += w[:-1] + " "
    return out

print("sai :", repr(wrong_concat("is2 This1")), "| đúng:", repr(Solution().sortSentence("is2 This1")))

sai : 'This is ' | đúng: 'This is'


Dấu cách cuối khiến so sánh chuỗi thất bại. Sửa: dùng `" ".join(...)` (tự chỉ chen **giữa**), hoặc `out.strip()`.

## 13. Mẹo nhận dạng pattern

- Mỗi phần tử **mang sẵn vị trí đích** (số thứ tự, rank, id liên tục 1..n) ⇒ **đặt thẳng vào ô** (`res[pos] = ...`), `O(n)`, không cần sort.
- Cần sắp theo một thuộc tính ⇒ `sorted(..., key=lambda x: thuộc_tính)`.
- Thao tác trên từ của câu ⇒ `split()` → xử lý list → `' '.join(...)`.
- Cắt đuôi cố định ⇒ `w[:-k]`; lấy đuôi ⇒ `w[-k:]`.
- Nếu số thứ tự có thể **nhiều chữ số** ⇒ tách phần chữ số cuối bằng vòng lặp hoặc `rstrip` (xem biến thể).

## 14. Biến thể

1. **Số thứ tự nhiều chữ số** (câu > 9 từ): tách phần số ở cuối mỗi từ.
2. **Xáo trộn câu** (bài ngược): từ câu gốc tạo câu đã gắn số.
3. **Số thứ tự đặt ở đầu từ**: `"1This 2is"`.
4. **LeetCode 1528 — Shuffle String**: `s` và `indices`, ký tự `s[i]` về vị trí `indices[i]` — cùng ý tưởng "đặt thẳng".

In [23]:
def sort_sentence_multi_digit(s):
    pairs = []
    for w in s.split():
        word = w.rstrip("0123456789")          # bỏ mọi chữ số ở cuối
        num = int(w[len(word):])
        pairs.append((num, word))
    pairs.sort()
    return " ".join(word for _, word in pairs)

def restore_string(s, indices):
    res = [""] * len(s)
    for ch, i in zip(s, indices):
        res[i] = ch
    return "".join(res)

print(sort_sentence_multi_digit("k11 a1 j10 b2"))
print(restore_string("codeleet", [4, 5, 6, 7, 0, 2, 1, 3]))

a b j k
leetcode


`w.rstrip(chars)` bỏ mọi ký tự thuộc `chars` ở **cuối** chuỗi.

## 15. Bài tập (tự làm trước)

**Bài 1 — tính tay:** khôi phục `"good3 day1 a2 have4 Nice0"`? (Cẩn thận: câu này có **hợp lệ** theo đề không?) Rồi khôi phục `"loves2 Python3 Everyone1"`.

**Bài 2 — dự đoán output:**

```python
w = "Python3"
print(w[-1], w[:-1], int(w[-1]) - 1, w[-1] * 2)
```

**Bài 3 — điền TODO:** đặt thẳng vào vị trí.

In [24]:
def exercise_sort_sentence(s):
    # TODO 1: words = s.split()
    # TODO 2: res = [""] * len(words)
    # TODO 3: res[int(w[-1]) - 1] = w[:-1]
    # TODO 4: return " ".join(res)
    pass

# assert exercise_sort_sentence("is2 sentence4 This1 a3") == "This is a sentence"
# assert exercise_sort_sentence("Myself2 Me1 I4 and3") == "Me Myself and I"

**Bài 4 — bài ngược:** viết `exercise_shuffle(sentence, order)` gắn số thứ tự (từ 1) vào mỗi từ rồi xếp các từ theo list chỉ số `order` (một hoán vị của `0..n-1`).

In [25]:
def exercise_shuffle(sentence, order):
    # TODO: tagged = [w + str(i + 1) ...]; trả " ".join(tagged[i] for i in order)
    pass

# assert exercise_shuffle("This is a sentence", [1, 3, 0, 2]) == "is2 sentence4 This1 a3"

**Bài 5 — LeetCode 1528 Shuffle String:** `restore(s, indices)` trả chuỗi mà ký tự `s[i]` nằm ở vị trí `indices[i]`.

In [26]:
def exercise_restore_string(s, indices):
    # TODO: đặt thẳng vào vị trí
    pass

# assert exercise_restore_string("codeleet", [4, 5, 6, 7, 0, 2, 1, 3]) == "leetcode"
# assert exercise_restore_string("abc", [0, 1, 2]) == "abc"

**Bài 6 — mini challenge:** số thứ tự đặt ở **đầu** từ và có thể nhiều chữ số, ví dụ `"2is 10end 1This"` → `"This is end"` (số không cần liên tục, chỉ cần tăng dần).

In [27]:
def exercise_sort_prefix_numbers(s):
    # TODO: lstrip chữ số để tách phần chữ; phần số = w[:len(w) - len(word)]
    pass

# assert exercise_sort_prefix_numbers("2is 10end 1This") == "This is end"
# assert exercise_sort_prefix_numbers("3c 1a 2b") == "a b c"

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | số đề bắt đầu từ 1 | `0` không hợp lệ | câu 2: `Everyone loves Python` |
| 2 | `w[-1]` là chuỗi | `int` rồi trừ 1 | `'3' * 2 = '33'` |
| 3 | mỗi từ có địa chỉ | ô `int(w[-1]) - 1` | `" ".join(res)` |
| 4 | gắn số theo vị trí gốc | `w + str(i + 1)` | lấy `tagged[i]` theo `order` |
| 5 | `res = [""] * len(s)` | `zip(s, indices)` | `res[i] = ch` |
| 6 | `lstrip("0123456789")` | tách `(num, word)` | sắp theo `num` rồi ghép |

## 17. Đáp án tham khảo

**Bài 1:** câu đầu **không hợp lệ** vì có `0` (đề chỉ có 1..9) — với 5 từ, cách đặt vị trí sẽ ghi `Nice` vào ô `-1` (ô cuối!) gây kết quả sai lệch. Câu 2 → `"Everyone loves Python"`.

**Bài 2:** in `3 Python 2 33`.

In [28]:
print(Solution().sortSentence("loves2 Python3 Everyone1"))
w = "Python3"
print(w[-1], w[:-1], int(w[-1]) - 1, w[-1] * 2)
print("câu không hợp lệ cho kết quả lệch:", repr(Solution().sortSentence("good3 day1 a2 have4 Nice0")))

Everyone loves Python
3 Python 2 33
câu không hợp lệ cho kết quả lệch: 'day a good have Nice'


In [29]:
def answer_sort_sentence(s):
    words = s.split()
    res = [""] * len(words)
    for w in words:
        res[int(w[-1]) - 1] = w[:-1]
    return " ".join(res)


def answer_shuffle(sentence, order):
    tagged = [w + str(i + 1) for i, w in enumerate(sentence.split())]
    return " ".join(tagged[i] for i in order)


def answer_restore_string(s, indices):
    res = [""] * len(s)
    for ch, i in zip(s, indices):
        res[i] = ch
    return "".join(res)


def answer_sort_prefix_numbers(s):
    pairs = []
    for w in s.split():
        word = w.lstrip("0123456789")
        num = int(w[: len(w) - len(word)])
        pairs.append((num, word))
    pairs.sort()
    return " ".join(word for _, word in pairs)


assert answer_sort_sentence("is2 sentence4 This1 a3") == "This is a sentence"
assert answer_sort_sentence("Myself2 Me1 I4 and3") == "Me Myself and I"
assert answer_shuffle("This is a sentence", [1, 3, 0, 2]) == "is2 sentence4 This1 a3"
assert answer_restore_string("codeleet", [4, 5, 6, 7, 0, 2, 1, 3]) == "leetcode"
assert answer_restore_string("abc", [0, 1, 2]) == "abc"
assert answer_sort_prefix_numbers("2is 10end 1This") == "This is end"
assert answer_sort_prefix_numbers("3c 1a 2b") == "a b c"
for _ in range(200):
    n = rng.randint(1, 9)
    sentence = " ".join("".join(rng.choice(string.ascii_letters) for _ in range(rng.randint(1, 5))) for _ in range(n))
    order = list(range(n)); rng.shuffle(order)
    assert answer_sort_sentence(answer_shuffle(sentence, order)) == sentence
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def sortSentence(self, s: str) -> str:
        words = s.split()
        res = [""] * len(words)
        for w in words:
            res[int(w[-1]) - 1] = w[:-1]     # ký tự cuối = vị trí (từ 1), phần trước = từ gốc
        return " ".join(res)
```

Câu nhớ: **"Biết địa chỉ thì đặt thẳng, khỏi sắp xếp."**

Checklist:

- [ ] Bỏ chữ số: `w[:-1]`.
- [ ] Vị trí = `int(w[-1]) - 1` (đề đếm từ 1).
- [ ] Ghép bằng `" ".join(...)` — không thừa dấu cách.
- [ ] Không dùng sắp xếp mặc định (theo chữ cái).
- [ ] Nếu đề khác cho số nhiều chữ số ⇒ không dùng `w[-1]` đơn thuần.